# ভিশন এনকোডার ও ইমেজ টোকেনাইজেশন

ভিশন এনকোডার এবং ইমেজ টোকেনাইজেশন — scratch থেকে তৈরি একটি ViT patch tokenizer।

কোনো download নেই, কোনো pretrained weight নেই, কোনো internet নেই। এখানের সবকিছু CPU-তে কৃত্রিম tensor-এর উপর কয়েক সেকেন্ডে চলে, কিন্তু মুদ্রিত প্রতিটি সংখ্যা সত্যিই হিসাব করা হয়, কেবল assert করা নয়।

এই স্ক্রিপ্টটি যা প্রদর্শন করে, ক্রমানুসারে:

  1. Patch embedding: একটি `(C, H, W)` ইমেজ tensor-কে `(N_patches, d_model)` vector-এর sequence-এ রূপান্তর করা একটি মাত্র strided Conv2d দিয়ে — প্রতিটি ViT-পরিবারের vision tower যে নির্ভুল অপারেশন দিয়ে শুরু হয়।
  2. Token-সংখ্যা বিস্ফোরণ: N_patches resolution-এর বর্গ হারে বাড়ে, আর attention খরচ N_patches-এর বর্গ হারে বাড়ে, তাই vision-tower খরচ ইমেজ বাহু-দৈর্ঘ্যের চতুর্থ ঘাত হারে বাড়ে। এখানে তা প্রকৃত token সংখ্যা ও প্রকৃত attention-matrix উপাদান সংখ্যা হিসাবে মাপা হয়েছে।
  3. 2D positional embedding interpolation: 224x224-এ pretrained একটি ViT-এর নির্দিষ্ট-আকারের position grid আছে; 448x448-এ চালাতে হলে সেই grid-কে bicubically resize করা প্রয়োজন। আমরা তা করি এবং round-trip error মাপি, দেখাই কেন এটি আদৌ কাজ করে।
  4. Dynamic tiling ("AnyRes"): interpolation-এর production বিকল্প — একটি বড় ইমেজকে 224x224 tile-এ কাটো, প্রতিটি tile native scale-এ encode করো, আর concatenate করো। আমরা উভয় কৌশলের token/attention বাজেট হিসাব করি।
  5. Pooling কৌশল: CLS token বনাম patch token-এর উপর mean pooling, আর কেন একটি VLM pooling করে একটি vector-এ নামানোর বদলে সব patch token-ই রাখে।

চালানোর পদ্ধতি:
```
python example.py
```
অথবা notebook-এ কোষগুলো উপর থেকে নিচ পর্যন্ত চালান — প্রতিটি অংশের কোষ তার নিজস্ব ডেমো চালায়।

In [ ]:
"""ভিশন এনকোডার ও ইমেজ টোকেনাইজেশন -- scratch থেকে তৈরি একটি ViT patch tokenizer।

কোনো download নেই, কোনো pretrained weight নেই, কোনো internet নেই। এখানের সবকিছু CPU-তে
কৃত্রিম tensor-এর উপর কয়েক সেকেন্ডে চলে, কিন্তু মুদ্রিত প্রতিটি সংখ্যা সত্যিই হিসাব করা
হয়, কেবল assert করা নয়।

এই স্ক্রিপ্টটি যা প্রদর্শন করে, ক্রমানুসারে:

  1. Patch embedding: একটি (C, H, W) ইমেজ tensor-কে (N_patches, d_model)
     vector-এর sequence-এ রূপান্তর একটি মাত্র strided Conv2d দিয়ে -- প্রতিটি
     ViT-পরিবারের vision tower যে নির্ভুল অপারেশন দিয়ে শুরু হয়।
  2. Token-সংখ্যা বিস্ফোরণ: N_patches resolution-এর SQUARE হারে বাড়ে,
     এবং attention খরচ N_patches-এর বর্গ হারে বাড়ে, তাই vision-tower
     খরচ ইমেজ বাহু-দৈর্ঘ্যের FOURTH power হারে বাড়ে। এখানে তা প্রকৃত token
     সংখ্যা ও প্রকৃত attention-matrix উপাদান সংখ্যা হিসেবে মাপা হয়েছে।
  3. 2D positional embedding interpolation: 224x224-এ pretrained একটি ViT-এর
     নির্দিষ্ট আকারের position grid থাকে; 448x448-এ চালাতে সেই grid-কে bicubically
     resize করা লাগে। আমরা তা করি এবং round-trip error মাপি, দেখাই
     কেন এটি আদৌ কাজ করে।
  4. Dynamic tiling ("AnyRes"): interpolation-এর production বিকল্প --
     একটি বড় ইমেজকে 224x224 tile-এ কাটো, প্রতিটি native scale-এ encode করো, আর
     concatenate করো। আমরা উভয় কৌশলের token/attention বাজেট হিসাব করি।
  5. Pooling কৌশল: CLS token বনাম patch token-এর উপর mean pooling, এবং কেন একটি
     VLM একটি vector-এ pool না করে সব patch token রেখে দেয়।

চালানোর পদ্ধতি:
    python example.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

## 1. Patch embedding: সম্পূর্ণ "ইমেজ টোকেনাইজার"

প্রতিটি ViT বাস্তবায়নের কৌশল: kernel size সমান stride বিশিষ্ট একটি Conv2d আক্ষরিক অর্থেই "প্রতিটি patch flatten করো, তারপর একটি shared Linear প্রয়োগ করো"। কোনো convolutional inductive bias বাকি থাকে না — patchগুলো কখনো overlapping হয় না।

In [ ]:
class PatchEmbed(nn.Module):
    """একটি ইমেজকে অসম্পন্ন patch-এ কাটো এবং প্রতিটি patch-কে linearly project করো।

    প্রতিটি ViT বাস্তবায়ন যে কৌশল ব্যবহার করে: একটি Conv2d যার kernel size তার stride-এর
    সমান — সেটিই আক্ষরিকভাবে "প্রতিটি patch flatten করো, তারপর একটি shared Linear প্রয়োগ
    করো"। কোনো convolutional inductive bias বাকি থাকে না -- patchগুলো কখনোই overlapping হয় না।
    """

    def __init__(self, image_size=224, patch_size=16, in_channels=3, d_model=768):
        super().__init__()
        assert image_size % patch_size == 0, "image size must be divisible by patch size"
        self.image_size = image_size
        self.patch_size = patch_size
        self.grid = image_size // patch_size          # প্রতি বাহুতে patch
        self.num_patches = self.grid * self.grid
        self.proj = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        # প্রতিটি grid cell-এর জন্য একটি শেখা position vector (এখানে CLS position নেই)।
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, d_model) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

    def forward(self, images):
        # images: (B, C, H, W)
        x = self.proj(images)                          # (B, d_model, grid, grid)
        x = x.flatten(2).transpose(1, 2)               # (B, num_patches, d_model)
        x = x + self.pos_embed                         # position যোগ করা হয়, যেমন Phase 02-তে
        cls = self.cls_token.expand(x.shape[0], -1, -1)
        return torch.cat([cls, x], dim=1)              # (B, 1 + num_patches, d_model)


print("=" * 74)
print("1. PATCH EMBEDDING: an image becomes a sequence of vectors")
print("=" * 74)

D_MODEL = 384
embed = PatchEmbed(image_size=224, patch_size=16, d_model=D_MODEL)
images = torch.randn(2, 3, 224, 224)                   # 2টি ভুয়া ইমেজের একটি batch
tokens = embed(images)

print(f"input  images shape : {tuple(images.shape)}  (B, C, H, W) -- raw pixels")
print(f"output tokens shape : {tuple(tokens.shape)}  (B, 1+N_patches, d_model)")
print(f"patch grid          : {embed.grid} x {embed.grid} = {embed.num_patches} patch tokens (+1 CLS)")
print(f"pixels per patch    : 16x16x3 = {16 * 16 * 3} numbers -> {D_MODEL}-dim vector")
print()
print("Sanity check: the Conv2d really is 'flatten patch, then one shared Linear'.")
patch_00 = images[0, :, 0:16, 0:16].reshape(-1)                       # প্রথম patch, flattened
w = embed.proj.weight.reshape(D_MODEL, -1)                            # (d_model, C*p*p)
manual = w @ patch_00 + embed.proj.bias
conv_out = embed.proj(images)[0, :, 0, 0]
print(f"  max |manual_linear - conv2d| = {(manual - conv_out).abs().max().item():.3e}  (~0 => identical op)")
print()

## 2. খরচ বক্ররেখা: token বাড়ে resolution^2 হারে, attention বাড়ে resolution^4 হারে

নির্দিষ্ট patch আকারে token সংখ্যা ইমেজ বাহু-এর বর্গ হারে বৃদ্ধি পায়, আর tower-এর ভেতরের self-attention সেটির বর্গ হারে বৃদ্ধি পায়।

In [ ]:
print("=" * 74)
print("2. WHY RESOLUTION IS EXPENSIVE (real counts, patch=16)")
print("=" * 74)
print(f"{'resolution':>12} {'patch tokens':>14} {'attn matrix cells':>20} {'rel. attn cost':>16}")
base_attn = None
for res in [224, 336, 448, 672, 896, 1344]:
    grid = res // 16
    n = grid * grid
    attn_cells = n * n                        # একটি head, একটি layer: N x N score
    if base_attn is None:
        base_attn = attn_cells
    print(f"{res:>9}px {n:>14,} {attn_cells:>20,} {attn_cells / base_attn:>15.1f}x")
print()
print("Token count scales with res^2; self-attention with (res^2)^2 = res^4.")
print("Going 224 -> 896 (4x the side) costs 256x the attention work in the")
print("vision tower AND puts 16x more tokens into the LLM's context window.")
print("That second cost is usually the one that hurts: see Lessons 4 and 10.")
print()

## 3. নতুন resolution-এর জন্য positional-embedding interpolation

শেখা absolute position grid-গুলোর একটি নির্দিষ্ট সমস্যা আছে: 224px-এ pretrained tower-এর কাছে ঠিক 196টি position vector আছে। 448px-এ চালালে দরকার 784টি। সর্বজনীন সমাধান — grid-টিকে bicubically interpolate করা।

In [ ]:
def interpolate_pos_embed(pos_embed, old_grid, new_grid):
    """একটি (1, old_grid^2, d) position grid-কে bicubically (1, new_grid^2, d)-এ resize করো।"""
    d = pos_embed.shape[-1]
    grid = pos_embed.reshape(1, old_grid, old_grid, d).permute(0, 3, 1, 2)   # (1, d, g, g)
    grid = F.interpolate(grid, size=(new_grid, new_grid), mode="bicubic", align_corners=False)
    return grid.permute(0, 2, 3, 1).reshape(1, new_grid * new_grid, d)


print("=" * 74)
print("3. RUNNING A 224px-PRETRAINED TOWER AT 448px: position interpolation")
print("=" * 74)

old_grid, new_grid = 14, 28
pos_224 = embed.pos_embed.detach()
pos_448 = interpolate_pos_embed(pos_224, old_grid, new_grid)
print(f"pretrained position grid : {tuple(pos_224.shape)}  ({old_grid}x{old_grid} = {old_grid ** 2} positions)")
print(f"interpolated to          : {tuple(pos_448.shape)}  ({new_grid}x{new_grid} = {new_grid ** 2} positions)")

# Round trip: সঙ্কুচিত করে আবার তুলনা করো। একটি *র্যান্ডম* position grid হল high-frequency
# noise, interpolation-এর জন্য সবচেয়ে খারাপ সম্ভাব্য ক্ষেত্র; একটি প্রশিক্ষিত grid অনেক মসৃণ।
# দুটোই দেখানো হয় এই বক্তব্য প্রতিষ্ঠার জন্য যে interpolation-কে সম্ভব করে তোলে মসৃণতা --
# resize করা নিজেই নয়।
round_trip = interpolate_pos_embed(pos_448, new_grid, old_grid)
err = (round_trip - pos_224).abs().mean().item()
scale = pos_224.abs().mean().item()
print(f"random (untrained) grid, round-trip 14->28->14 relative error : {100 * err / scale:5.2f}%")

# এখন একটি মসৃণ grid, যা একটি প্রশিক্ষিত grid-এর স্থলে দাঁড়ায়: প্রকৃত শেখা 2D position
# grid-গুলো দৃঢ়ভাবে স্থানিকভাবে সম্পর্কযুক্ত (প্রতিবেশী position-গুলো একই রকম vector পায়),
# যা interpolation-এর ঠিকই প্রয়োজন।
smooth = F.avg_pool2d(
    F.pad(pos_224.reshape(1, old_grid, old_grid, -1).permute(0, 3, 1, 2), (1, 1, 1, 1), mode="replicate"),
    kernel_size=3, stride=1,
).permute(0, 2, 3, 1).reshape(1, old_grid ** 2, -1)
smooth_rt = interpolate_pos_embed(interpolate_pos_embed(smooth, old_grid, new_grid), new_grid, old_grid)
serr = (smooth_rt - smooth).abs().mean().item() / smooth.abs().mean().item()
print(f"smooth  (trained-like) grid, same round trip relative error   : {100 * serr:5.2f}%")
print("Smoothness is the empirical reason interpolation works at all; it is still")
print("an approximation, which is why models are usually fine-tuned briefly at the")
print("new resolution afterwards.")
print()

## 4. Dynamic tiling (AnyRes) বনাম একটি বড় interpolated forward pass

Tiling token সংখ্যা কমানোয় না, কিন্তু প্রতিটি tile-কে এনকোডারের native প্রশিক্ষিত resolution-এ রাখে — position-interpolation mismatch সম্পূর্ণ দূর করে — ট্রেড-অফ হলো, ভিন্ন tile-এর patchগুলো tower-এর ভেতরে কখনো একে অপরের প্রতি attend করে না।

In [ ]:
print("=" * 74)
print("4. TWO WAYS TO HANDLE A 896x896 IMAGE (patch=16, base tower 224px)")
print("=" * 74)

res, tile = 896, 224
n_tiles = (res // tile) ** 2
tokens_per_tile = (tile // 16) ** 2

single_pass_tokens = (res // 16) ** 2
single_pass_attn = single_pass_tokens ** 2
tiled_tokens = n_tiles * tokens_per_tile + tokens_per_tile   # + একটি downscaled global "thumbnail" tile
tiled_attn = (n_tiles + 1) * (tokens_per_tile ** 2)

print("A) one forward pass at 896px (interpolated positions)")
print(f"   vision tokens   : {single_pass_tokens:,}")
print(f"   attention cells : {single_pass_attn:,}")
print("   every patch attends to every other patch (full global context)")
print()
print(f"B) tiled: {n_tiles} x {tile}px tiles + 1 downscaled global thumbnail")
print(f"   vision tokens   : {tiled_tokens:,}  (same order -- tokens are NOT saved)")
print(f"   attention cells : {tiled_attn:,}  ({single_pass_attn / tiled_attn:.1f}x cheaper)")
print("   each tile is encoded at the tower's NATIVE resolution -- no position")
print("   interpolation, no train/test mismatch -- but patches in different tiles")
print("   never attend to each other inside the tower, so the LLM has to stitch")
print("   the tiles together itself. The thumbnail tile is what gives it the")
print("   global layout.")
print()

## 5. Pooling: একটি classifier যা রাখে বনাম একটি VLM যা রাখে

Pooling অবস্থান প্রায় সম্পূর্ণভাবে ভেঙে দেয়; token sequence সেটি ধরে রাখে। CLIP-এর contrastive loss-এর (Lesson 2) প্রতি ইমেজে কেবল একটি vector প্রয়োজন, তাই তার training signal-কে কখনো অবস্থান সংরক্ষণ করতে হয় না।

In [ ]:
print("=" * 74)
print("5. POOLING: one vector (CLIP) vs. all patch tokens (VLM)")
print("=" * 74)

cls_vec = tokens[:, 0, :]                 # (B, d) -- CLS summary
mean_vec = tokens[:, 1:, :].mean(dim=1)   # (B, d) -- mean-pooled patches
all_patches = tokens[:, 1:, :]            # (B, N, d) -- সবকিছু

print(f"CLS pooled       : {tuple(cls_vec.shape)}   -> {cls_vec.numel():>7,} numbers/batch")
print(f"mean pooled      : {tuple(mean_vec.shape)}   -> {mean_vec.numel():>7,} numbers/batch")
print(f"all patch tokens : {tuple(all_patches.shape)} -> {all_patches.numel():>7,} numbers/batch")
print(f"ratio            : a VLM carries {all_patches.numel() / cls_vec.numel():.0f}x more information forward")
print()

# একটি concretely প্রদর্শন যে pooling অবস্থান তথ্য ধ্বংস করে:
# একই বিষয়বস্তু ভিন্ন জায়গায় ধারণকারী দুইটি ইমেজ।
img_a = torch.zeros(1, 3, 224, 224)
img_a[:, :, :112, :112] = 1.0                 # উজ্জ্বল বর্গ, উপরে-বামে
img_b = torch.zeros(1, 3, 224, 224)
img_b[:, :, 112:, 112:] = 1.0                 # একই বর্গ, নিচে-ডানে

with torch.no_grad():
    ta, tb = embed(img_a), embed(img_b)
pe = embed.pos_embed.detach()
seq_a, seq_b = ta[:, 1:], tb[:, 1:]
# position term সরান যাতে আমরা বিচ্ছিন্ন করি শুধু POOLING-ই কী হারায়।
mean_a_np = (seq_a - pe).mean(1)
mean_b_np = (seq_b - pe).mean(1)

print("Two images: an identical bright square, top-left vs bottom-right.")
print(f"  cosine(mean-pooled, position term removed) : {F.cosine_similarity(mean_a_np, mean_b_np).item():.4f}  <- indistinguishable")
print(f"  fraction of patch tokens that differ       : {(seq_a - seq_b).abs().sum(-1).gt(1e-6).float().mean().item():.0%}")
print()
print("Mean pooling collapses 'where' almost entirely; the token sequence keeps it.")
print("CLIP's contrastive loss (Lesson 2) only ever needs ONE vector per image, so")
print("its training signal never has to preserve location. A VLM asked 'what is")
print("written on the left-hand sign?' needs per-location detail, so it feeds the")
print("whole patch sequence to the LLM -- which is precisely why vision tokens")
print("dominate a VLM's context budget, and why Lesson 4 is about compressing them")
print("without throwing that detail away.")